# Nuvoton Industry Project -- Elevator People Counting on the M55M1

This notebook covers the following steps to calibrate and export the YOLO model onto monochrome device specifications:
1. **Conda environment setup** running inside Colab with `condacolab`.
2. **Dataset downloads and layouts** from HuggingFace to a 90/10 layout split.
3. **Local training running** with constant 192x192 monochrome variables triggers inputs.
4. **Evaluation metrics validation** layout scripts.
5. **Onnx/TFLite layout conversion exports** matching Ethos-U55 parameters.

In [1]:
!pip install -q condacolab
import condacolab
condacolab.install_from_url("https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-Linux-x86_64.sh")

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:08
🔁 Restarting kernel...


In [1]:
!conda --version

conda 25.3.1


In [2]:
GIT_REPO_URL = "https://github.com/bencejdanko/m55m1-ElevatorCounting-YOLOv8n"
!git clone {GIT_REPO_URL}

import os
dir_name = os.path.basename(GIT_REPO_URL).replace('.git', '')
%cd {dir_name}/yolov8_ultralytics

Cloning into 'm55m1-ElevatorCounting-YOLOv8n'...
remote: Enumerating objects: 947, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 947 (delta 36), reused 80 (delta 22), pack-reused 845 (from 2)
Receiving objects: 100% (947/947), 151.18 MiB | 38.30 MiB/s, done.
Resolving deltas: 100% (236/236), done.
/content/m55m1-ElevatorCounting-YOLOv8n/yolov8_ultralytics


In [3]:
!conda create --name yolov8_DG python=3.10 -y

!conda run -n yolov8_DG python -m pip install --upgrade pip setuptools
!conda run -n yolov8_DG python -m pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu118

# Install current local package using .[export]
!conda run -n yolov8_DG python -m pip install .[export]

# Additional supporting packages needed for download/export
!conda run -n yolov8_DG python -m pip install datasets onnx2tf onnx
!conda run -n yolov8_DG python -m pip install matplotlib
!conda run -n yolov8_DG python -m pip install py-cpuinfo

Channels:
 - conda-forge
Platform: linux-64
Solving environment: | done


==> WARNING: A newer version of conda exists. <==
    current version: 25.3.1
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/yolov8_DG

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-1.0.8                |       hda65f42_9         254 KB  conda-forge
    ca-certificates-2026.2.25  |       hbd8a1cb_0         144 KB  conda-forge
    icu-78.3                   |       h33c6efd_0        12.1 MB  conda-forge
    ld_impl_linux-64-2.45.1    |default_hbd61a6d_102         711 KB  conda-forge
    libexpat-2.7.4             |       hecca717_0          75 KB  conda-forge
    

In [4]:
# Download bdanko/overhead-person-detection dataset
# more info https://github.com/bencejdanko/m55m1-ElevatorCounting-YOLOv8n
!conda run -n yolov8_DG python download_hf_dataset.py --dataset bdanko/overhead-person-detection --out-dir datasets/overhead_monochrome

Loading HuggingFace dataset: bdanko/overhead-person-detection
Total items: 13448
Populating local environment folder splits...
Processing split: train (12103 items)
Processing split: val (1345 items)

Done! Dataset structure created at: /content/m55m1-ElevatorCounting-YOLOv8n/yolov8_ultralytics/datasets/overhead_monochrome
You can train with: --data datasets/overhead_monochrome/data.yaml


Generating train split: 100%|██████████| 13448/13448 [00:00<00:00, 14465.70 examples/s]

100%|██████████| 12103/12103 [00:44<00:00, 269.60it/s]

100%|██████████| 1345/1345 [00:05<00:00, 239.98it/s]



In [9]:
!conda run -n yolov8_DG env MPLBACKEND=Agg python dg_train.py \
  --model-cfg relu6-yolov8.yaml \
  --data datasets/overhead_monochrome/data.yaml \
  --imgsz 192 \
  --weights yolov8n.pt \
  --epochs 10 \
  --device 0

WARNING ⚠️ no model scale passed. Assuming scale='n'.
Transferred 355/451 items from pretrained weights
{'cfg': None, 'data': 'datasets/overhead_monochrome/data.yaml', 'epochs': 10, 'batch': 64, 'imgsz': 192, 'device': ['0'], 'optimizer': 'SGD', 'workers': 2, 'project': 'runs/train', 'name': 'exp', 'exist_ok': False, 'patience': 0, 'cache': False, 'close_mosaic': 3, 'resume': False, 'lr0': 0.01, 'lrf': 0.01, 'cos_lr': False, 'scale': 0.5, 'mixup': 0.0, 'copy_paste': 0.0}
New https://pypi.org/project/ultralytics/8.4.26 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.42 🚀 Python-3.10.20 torch-2.5.1+cu118 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=relu6-yolov8.yaml, data=datasets/overhead_monochrome/data.yaml, epochs=10, time=None, patience=0, batch=64, imgsz=192, save=True, save_period=-1, cache=False, device=['0'], workers=2, project=runs/train, name=exp, exist_ok=False, pretrained=True, optimizer=SGD, verbose=True, seed=0, deter

In [11]:
# validation
# Note the experimental run you saved at!
EXPERIMENT="exp"
!conda run -n yolov8_DG env MPLBACKEND=Agg python dg_val.py \
  --weights runs/train/{EXPERIMENT}/weights/best.pt \
  --data datasets/overhead_monochrome/data.yaml \
  --imgsz 192 \
  --device 0

{'weights': 'runs/train/exp/weights/best.pt', 'data': 'datasets/overhead_monochrome/data.yaml', 'annotations': None, 'device': '0', 'imgsz': 192, 'no_separate_outputs': False}
Ultralytics YOLOv8.2.42 🚀 Python-3.10.20 torch-2.5.1+cu118 CUDA:0 (Tesla T4, 14913MiB)
relu6-YOLOv8 summary (fused): 200 layers, 3278259 parameters, 0 gradients
                   all       1345       2868      0.932      0.855      0.935      0.548
WARNING ⚠️ ConfusionMatrix plot failure: No module named 'seaborn'
WARNING ⚠️ ConfusionMatrix plot failure: No module named 'seaborn'
Speed: 0.0ms preprocess, 1.2ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /content/m55m1-ElevatorCounting-YOLOv8n/runs/detect/val


val: Scanning /content/m55m1-ElevatorCounting-YOLOv8n/yolov8_ultralytics/datasets/overhead_monochrome/val/labels.cache... 1345 images, 55 backgrounds, 0 corrupt: 100%|██████████| 1345/1345 [00:00<?, ?it/s]
val: Scanning /content/m55m1-ElevatorCounting-YOLOv8n/yolov8_ultralytics/data

In [13]:
# graph intermediary to onnx vectors
!conda run -n yolov8_DG env MPLBACKEND=Agg python nu_export_tflite_int8.py --format onnx --weights ./runs/train/{EXPERIMENT}/weights/best.pt --img 192

{'format': 'onnx', 'weights': './runs/train/exp/weights/best.pt', 'imgsz': 192, 'device': 'cpu', 'img_dir': None, 'n_img': 200, 'out': 'calib_data.npy'}
Ultralytics YOLOv8.2.42 🚀 Python-3.10.20 torch-2.5.1+cu118 CPU (Intel Xeon 2.00GHz)
relu6-YOLOv8 summary (fused): 200 layers, 3278259 parameters, 0 gradients

PyTorch: starting from 'runs/train/exp/weights/best.pt' with input shape (1, 3, 192, 192) BCHW and output shape(s) ((1, 576, 64), (1, 144, 64), (1, 36, 64), (1, 576, 1), (1, 144, 1), (1, 36, 1)) (6.5 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.31'] not found, attempting AutoUpdate...


requirements: AutoUpdate success ✅ 1.3s, installed 1 package: ['onnxslim>=0.1.31']
requirements: ⚠️ Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.19.1 opset 19...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 2.5s, saved as 'runs/train/exp/weights/best.onnx' (11.5 MB)

Export complete (3.8s)
Results saved to /content/m5

In [36]:
# graph intermediary to onnx vectors
!conda run -n yolov8_DG env MPLBACKEND=Agg python nu_export_tflite_int8.py --format onnx --weights ./runs/train/{EXPERIMENT}/weights/best.pt --img 192

{'format': 'onnx', 'weights': './runs/train/exp/weights/best.pt', 'imgsz': 192, 'device': 'cpu', 'img_dir': None, 'n_img': 200, 'out': 'calib_data.npy'}
Ultralytics YOLOv8.2.42 🚀 Python-3.10.20 torch-2.5.1+cu118 CPU (Intel Xeon 2.00GHz)
relu6-YOLOv8 summary (fused): 200 layers, 3278259 parameters, 0 gradients

PyTorch: starting from 'runs/train/exp/weights/best.pt' with input shape (1, 3, 192, 192) BCHW and output shape(s) ((1, 576, 64), (1, 144, 64), (1, 36, 64), (1, 576, 1), (1, 144, 1), (1, 36, 1)) (6.5 MB)

ONNX: starting export with onnx 1.19.1 opset 19...
ONNX: slimming with onnxslim 0.1.90...
ONNX: export success ✅ 1.1s, saved as 'runs/train/exp/weights/best.onnx' (11.5 MB)

Export complete (2.4s)
Results saved to /content/m55m1-ElevatorCounting-YOLOv8n/yolov8_ultralytics/runs/train/exp/weights
Predict:         yolo predict task=detect model=runs/train/exp/weights/best.onnx imgsz=192  
Validate:        yolo val task=detect model=runs/train/exp/weights/best.onnx imgsz=192 data=da

In [37]:
# calibrations referencing saved monochrome folder
!conda run -n yolov8_DG env MPLBACKEND=Agg python generate_calib_data.py \
    --img-size 192 192 --n-img 100 -o calib_data_192_n100.npy \
    --img-dir datasets/overhead_monochrome/train/images

12103
convert from BGR to RGB format!!
(100, 192, 192, 3)
calib_datas.shape: (100, 192, 192, 3)



In [38]:
!rm -rf saved_model
!conda run -n yolov8_DG env MPLBACKEND=Agg onnx2tf \
   -i runs/train/{EXPERIMENT}/weights/best.onnx \
   -oiqt \
   -nuo \
   -cind images calib_data_192_n100.npy "[[[[0]],[[0]],[[0]]]]" "[[[[1]],[[1]],[[1]]]]"


Automatic generation of each OP name started ========================================
Automatic generation of each OP name complete!

Model loaded ========================================================================

Model conversion started ============================================================
INFO: input_op_name: images shape: [1, 3, 192, 192] dtype: float32
 index: 1 Got: 192 Expected: 3
 index: 3 Got: 3 Expected: 192
 Please fix either the inputs/outputs or the model.

INFO: 2 / 173
INFO: onnx_op_type: Conv onnx_op_name: /model.0/conv/Conv
INFO:  input_name.1: images shape: [1, 3, 192, 192] dtype: float32
INFO:  input_name.2: model.0.conv.weight shape: [16, 3, 3, 3] dtype: float32
INFO:  input_name.3: model.0.conv.bias shape: [16] dtype: float32
INFO:  output_name.1: /model.0/conv/Conv_output_0 shape: [1, 16, 96, 96] dtype: float32
INFO: tf_op_type: convolution_v2
INFO:  input.1.input: name: tf.compat.v1.pad/Pad:0 shape: (1, 194, 194, 3) dtype: <dtype: 'float32'> 
INFO:

In [44]:
# Run onnx2tf to extract TFLite integers graph fully
!rm -rf saved_model
!conda run -n yolov8_DG env MPLBACKEND=Agg onnx2tf -i runs/train/{EXPERIMENT}/weights/best.onnx -oiqt -iqd int8 -oqd int8 -cind images calib_data_192_n4_rgb.npy "[[[[0,0,0]]]]" "[[[[1,1,1]]]]"


Model optimizing started ============================================================
Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Add        │ 6              │ 6                │
│ Clip       │ 65             │ 65               │
│ Concat     │ 13             │ 13               │
│ Constant   │ 147            │ 147              │
│ Conv       │ 71             │ 71               │
│ MaxPool    │ 3              │ 3                │
│ Reshape    │ 6              │ 6                │
│ Resize     │ 2              │ 2                │
│ Transpose  │ 6              │ 6                │
│ Model Size │ 11.5MiB        │ 11.5MiB          │
└────────────┴────────────────┴──────────────────┘

Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━

In [46]:
# confirming int8 export

import tensorflow as tf

model_path = "saved_model/best_full_integer_quant.tflite"
interpreter = tf.lite.Interpreter(model_path=model_path)

details = interpreter.get_input_details()[0]
print(f"Name:  {details['name']}")
print(f"Shape: {details['shape']}")
print(f"DType: {details['dtype']}")

Name:  serving_default_images:0
Shape: [  1 192 192   3]
DType: <class 'numpy.int8'>


In [47]:
!conda run -n yolov8_DG python -m pip install ethos-u-vela

In [49]:
!rm /content/best_integer_quant_vela.tflite
!conda run -n yolov8_DG vela saved_model/best_integer_quant.tflite \
    --accelerator-config ethos-u55-128 \
    --output-dir /content/ \
    --optimise Performance



Warning (supported operators) operator:DEQUANTIZE ofm:PartitionedCall:2
Reason: Unsupported opType

Warning (supported operators) operator:QUANTIZE ofm:tfl.quantize
Reason: Operation has tensor with unsupported DataType Float32
Feature-map dataTypes must be one of {"INT16", "INT32", "INT64", "INT8", "UINT8"}

Warning (supported operators) operator:DEQUANTIZE ofm:PartitionedCall:0
Reason: Unsupported opType

Warning (supported operators) operator:DEQUANTIZE ofm:PartitionedCall:3
Reason: Unsupported opType

Warning (supported operators) operator:DEQUANTIZE ofm:PartitionedCall:5
Reason: Unsupported opType

Warning (supported operators) operator:DEQUANTIZE ofm:PartitionedCall:1
Reason: Unsupported opType

Warning (supported operators) operator:DEQUANTIZE ofm:PartitionedCall:4
Reason: Unsupported opType
CPU performance estimation for "Passthrough" not implemented
CPU performance estimation for "Passthrough" not implemented
CPU performance estimation for "Passthrough" not implemented
CPU pe

In [42]:
!ls -lh /content/*_vela.tflite

-rw-r--r-- 1 root root 2.7M Mar 24 05:47 /content/best_integer_quant_vela.tflite
